# E-mail communication network Data processing

After downloading all datasets (as described in raw/emails/readme.txt) we proceed to process the data. 

We need to build the adjacency matrices for each snapshot.

In [ ]:
import pandas as pd
import warnings
import xlrd
import numpy as np
import seaborn as sns
warnings.filterwarnings('ignore')

In [ ]:
input_folder = "data/raw/emails/"
output_folder = "data/processed/emails/"

!mkdir -p data/processed/emails/

### Adjacency matrices

We want to build the adjacency matrices. $A^t_{ij}$ corresponds to the number of emails between user $i$ and $j$ at snapshot $t$. We store each snapshot independently in our output folder. Since we just have information of the upper diagonal, we will symmetrize these networks.

In order to reduce spurious links, we use filters for persistent communication (removing edges with less than 14 emails and removing users with less than 4 neighbours). Then, we take the intersection of remaining users as our user set for constructing the adjacency matrices.

In [ ]:
min_num_emails = 14
min_num_neighbours = 4

In [ ]:
years = ["07", "08", "09", "10"]

In [ ]:
list_matrices = []
list_userids_tokeep = []

for year in years:
    print(f"Processing year: {year}")
    file_name = f"{input_folder}/undweighted{year}.dat"
    df_A = pd.read_csv(file_name, delimiter=" ", header=None)
    df_A.columns = ["from", "to", "num_mails"]

    # Build adjacency matrix
    unique_users = np.sort(np.unique(np.concatenate([df_A["from"].values, df_A["to"].values])))
    dict_users = {uid: idx for idx, uid in enumerate(unique_users)}
    A = np.zeros((len(unique_users), len(unique_users)))

    for _, row in df_A.iterrows():
        i = dict_users[row['from']]
        j = dict_users[row['to']]
        weight = row['num_mails']

        if i < j:
            A[i, j] = weight
        else:
            A[j, i] = weight

    print("Sum of weights:", A.sum())

    #Threshold: min_num_emails 
    A[A <= min_num_emails] = 0.

    #Threshold: min_num_neighbours
    num_neighbours = (A > 0).sum(axis=0) + (A > 0).sum(axis=1)
    num_users_keep = np.where(num_neighbours > min_num_neighbours)[0]

    #Filter matrix
    A = A[np.ix_(num_users_keep, num_users_keep)]
    list_matrices.append(A)

    #Keep user IDs for filtered nodes
    inv_dict_users = {v: k for k, v in dict_users.items()}
    id_users_tokeep = [inv_dict_users[idx] for idx in num_users_keep]
    list_userids_tokeep.append(id_users_tokeep)

    print(f"Number of users after filtering: {len(id_users_tokeep)}")
    print("-------")

#Find nodes present in all snapshots
intersection_ids = set(list_userids_tokeep[0]).intersection(*list_userids_tokeep[1:])
ordered_ids = sorted(intersection_ids) #consistent order 

In [ ]:
print(f"N={len(intersection_ids)}")

In [ ]:
list_A = [] #we'll store the adjacency matrices of each snapshot with the same node ids
for y, year in enumerate(years):
    
    i_tokeep = [list_userids_tokeep[y].index(uid) for uid in ordered_ids]
    filtered_A = list_matrices[y][np.ix_(i_tokeep, i_tokeep)] #ensuring consistent order of the nodes in all snapshots 
    #symmetrize matrix:
    filtered_A = np.maximum(filtered_A, filtered_A.T)
    
    pd.DataFrame(filtered_A, index=ordered_ids, columns=ordered_ids).to_csv(f"{output_folder}/adjMat_{year}.csv")
    list_A.append(filtered_A)

### Groups and user metadata

In [ ]:
list_users_adjmat = list(intersection_ids)

In [ ]:
users_info = pd.read_csv(f"{input_folder}/usuaris.txt", delimiter=";", header=None)
users_info.columns = ["id", "role", "department_code"]

departments_info = pd.read_csv(f"{input_folder}/dept.txt", delimiter=";", header=None)
departments_info.columns = ["department_code", "department_name"]

users_info = pd.merge(users_info, departments_info, on='department_code', how='left')
users_info = (
    users_info[users_info["id"].isin(ordered_ids)]
    .set_index("id")
    .loc[ordered_ids]
    .reset_index()
)
users_info.head()

Based on the information provided by the university (https://www.urv.cat/ca/universitat/estructura/estructures/facultats-escoles/) we will associate each user to one faculty, based on the department. Note we don't have the department name from N1.101 (it appeared in the email networks but not in the department information). I will assume it's the same one as N101.ANT: Història, Història d'Art i Geografia.

In [ ]:
users_info.loc[users_info["department_code"] == "N1.101", "department_code"] = "N101.ANT"
users_info.loc[users_info["department_code"] == "N101.ANT", "department_name"] = "Història, Història de l'Art i Geografia"

In [ ]:
set(users_info["department_name"])

In [ ]:
dict_dept_faculty = {
    'Antropologia, Filosofia i Treball Social': "Lletres",
    'Bioquímica i Biotecnologia': "Química",
    'Ciències Mèdiques Bàsiques': "Medicina",
    'Dret Privat, Processal i Financer': "Jurídiques",
    'Dret Públic': "Jurídiques",
    'Economia': "Economia",
    'Eng. Electrònica, Elèctrica i Automàtica': "ETSE",
    'Enginyeria Informàtica i Matemàtiques': "ETSE",
    'Enginyeria Mecànica': "ETSEQ",
    'Enginyeria Química': "ETSEQ",
    'Estudis Anglesos i Alemanys': "Lletres",
    'Estudis de Comunicació': "Lletres",
    'Filologia Catalana': "Lletres",
    'Filologies Romàniques': "Lletres",
    'Geografia': "Turisme",
    "Gestió d'Empreses": "Economia",
    "Història, Història de l'Art i Geografia": "Lletres",
    'Infermeria': "Infermeria",
    "Institut de Ciències de l'Educació": "Educació",
    'Medicina i Cirurgia': "Medicina",
    'Pedagogia': "Educació",
    'Psicologia': "Educació",
    'Química Analítica i Química Orgànica': "Química",
    'Química Física i Inorgànica': "Química",
    "Unitat Predepartamental d'Arquitectura": "Arquitectura"
    }

In [ ]:
users_info["faculty"] = users_info["department_name"].map(dict_dept_faculty)
users_info

In [ ]:
print("Number of groups (faculties)=",len(set(users_info["faculty"])))

In [ ]:
users_info.to_csv(f"{output_folder}/user_metadata.csv")